In [11]:
import os
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI
import arxiv
import fitz
from deepagents import create_deep_agent
from langchain_core.tools import tool

load_dotenv()

True

In [12]:
def search_and_download_paper(query, download_path="../data/"):
    """Searches arXiv for a paper and downloads the PDF."""
    os.makedirs(download_path, exist_ok=True)
    search = arxiv.Search(query=query, max_results=1, sort_by=arxiv.SortCriterion.Relevance)
    paper = next(search.results())
    filename = f"{paper.title.replace(' ', '_').replace(':', '')}.pdf"
    path = paper.download_pdf(dirpath=download_path, filename=filename)
    return path

def extract_text_from_pdf(path):
    """Opens a PDF and extracts all text content."""
    doc = fitz.open(path)
    content = ""
    for page_num in range(len(doc)):
        content += f"\n--- Page {page_num + 1} ---\n"
        content += doc.load_page(page_num).get_text()
    return content

## Deep Agent Setup
This agent uses the `deepagents` framework to automate planning and research.

In [13]:
llm = ChatGoogleGenerativeAI(model="gemini-2.0-flash", temperature=0)

@tool
def research_arxiv(query: str):
    """Searches and downloads a paper from arXiv. Returns the path to the PDF."""
    return search_and_download_paper(query)

@tool
def extract_content(pdf_path: str):
    """Extracts all text from a PDF file."""
    return extract_text_from_pdf(pdf_path)

tools = [research_arxiv, extract_content]

agent = create_deep_agent(
    model=llm,
    tools=tools,
    system_prompt="You are a DeepForge Research Agent. Use your tools to find and analyze papers."
)

print("Deep Agent Initialized!")

Deep Agent Initialized!


In [14]:
# Example invocation
inputs = {"messages": [("user", "Research the QLoRA paper and summarize its findings.")]}
for chunk in agent.stream(inputs, config={"configurable": {"thread_id": "1"}}):
    print(chunk)

{'PatchToolCallsMiddleware.before_agent': {'messages': Overwrite(value=[HumanMessage(content='Research the QLoRA paper and summarize its findings.', additional_kwargs={}, response_metadata={}, id='71f8d794-718b-47e4-8535-6115b4c02693')])}}
{'model': {'messages': [AIMessage(content='', additional_kwargs={'function_call': {'name': 'research_arxiv', 'arguments': '{"query": "QLoRA"}'}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.0-flash', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019cf07b-51bb-7140-bdc2-c484766213d6-0', tool_calls=[{'name': 'research_arxiv', 'args': {'query': 'QLoRA'}, 'id': '4adefb08-f9af-4bc1-9eb6-01fb9d2ebed3', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 5197, 'output_tokens': 7, 'total_tokens': 5204, 'input_token_details': {'cache_read': 0}})]}}
{'TodoListMiddleware.after_model': None}


C:\Users\ASUS\AppData\Local\Temp\ipykernel_1760\1405917886.py:5: DeprecationWarning: The 'Search.results' method is deprecated, use 'Client.results' instead
  paper = next(search.results())


{'tools': {'messages': [ToolMessage(content='../data/Efficient_Telecom_Specific_LLM_TSLAM-Mini_with_QLoRA_and_Digital_Twin_Data.pdf', name='research_arxiv', id='4f6f1520-f789-42de-bdc2-0e6c341f1056', tool_call_id='4adefb08-f9af-4bc1-9eb6-01fb9d2ebed3')]}}
{'model': {'messages': [AIMessage(content='', additional_kwargs={'function_call': {'name': 'extract_content', 'arguments': '{"pdf_path": "../data/Efficient_Telecom_Specific_LLM_TSLAM-Mini_with_QLoRA_and_Digital_Twin_Data.pdf"}'}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.0-flash', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019cf07b-60a2-7f00-9f3d-149e67a260fe-0', tool_calls=[{'name': 'extract_content', 'args': {'pdf_path': '../data/Efficient_Telecom_Specific_LLM_TSLAM-Mini_with_QLoRA_and_Digital_Twin_Data.pdf'}, 'id': 'f9b92c7d-b8fe-4665-a863-143c8cbeefbd', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 5240, 'output_tokens': 38, 'total_tokens': 5278, 